In [19]:
import requests
from datetime import datetime, timedelta
import re
from zoneinfo import ZoneInfo # Python 3.9+ 표준 라이브러리

def get_latest_obs_time():
    """
    안정적인 조회를 위해 '한국 시간' 기준 1시간 전의 가장 최근 정각 시각을
    YYYYMMDDHHMM 형식으로 반환합니다.
    """
    now_kst = datetime.now(ZoneInfo("Asia/Seoul"))

    # ❗️ [진단 코드] 스크립트가 인식하는 현재 시간을 출력합니다.
    print(f"🕒 스크립트가 인식하는 현재 KST: {now_kst.strftime('%Y-%m-%d %p %I:%M:%S')}")

    one_hour_ago = now_kst - timedelta(hours=1)
    rounded = one_hour_ago.replace(minute=0, second=0, microsecond=0)
    return rounded.strftime("%Y%m%d%H%M")

def parse_surface_obs_data(text):
    """기상청 응답 텍스트에서 주요 관측값 추출 (공백 구분 형식 지원)"""
    lines = text.splitlines()
    data_line = None
    for line in lines:
        if line.startswith("#"):
            continue
        if line.strip():
            data_line = line
            break

    if data_line is None:
        print("❌ 관측 데이터 라인을 찾을 수 없습니다.")
        return

    parts = data_line.split()

    if len(parts) < 14:
        print("❌ 수신된 데이터 형식이 예상과 다릅니다.")
        return

    obs_time = parts[0]
    station_id = parts[1]
    wd = parts[2]
    ws = parts[3]
    ta = parts[11]
    td = parts[12]
    hm = parts[13]

    obs_time_fmt = datetime.strptime(obs_time, "%Y%m%d%H%M").strftime("%Y-%m-%d %H:%M")

    print("\n✅ 응답 수신 완료\n")
    print("📊 실황 관측 데이터")
    print(f"🕒 관측 시각: {obs_time_fmt} (KST)")
    print(f"📍 지점 번호: {station_id} (서울)")
    print(f"🌡️ 기온(TA): {ta} ℃")
    print(f"💧 이슬점(TD): {td} ℃")
    print(f"💦 습도(HM): {hm} %")
    print(f"🌬️ 풍향(WD): {wd}도 / 풍속(WS): {ws} m/s")

def get_kma_station_weather(stn=108, auth_key="여기에_인증키_입력"):
    """기상청 지점 기반 실황 데이터 조회"""
    tm = get_latest_obs_time()
    url = "https://apihub.kma.go.kr/api/typ01/url/kma_sfctm2.php"
    params = {
        "tm": tm,
        "stn": str(stn),
        "authKey": auth_key
    }

    print(f"📡 {tm} 기준 데이터 요청 중...")

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        parse_surface_obs_data(response.content.decode('utf-8'))
    except Exception as e:
        print(f"❌ 오류 발생: {e}")

# ✅ 실행 예시
if __name__ == "__main__":
    station_id = 108
    service_key = "4lCPsJEcQuSQj7CRHHLkDQ"
    get_kma_station_weather(stn=station_id, auth_key=service_key)


🕒 스크립트가 인식하는 현재 KST: 2025-07-28 PM 05:08:31
📡 202507281600 기준 데이터 요청 중...

✅ 응답 수신 완료

📊 실황 관측 데이터
🕒 관측 시각: 2025-07-28 16:00 (KST)
📍 지점 번호: 108 (서울)
🌡️ 기온(TA): 36.3 ℃
💧 이슬점(TD): 23.8 ℃
💦 습도(HM): 49.0 %
🌬️ 풍향(WD): 29도 / 풍속(WS): 2.5 m/s
